In [1]:
import joblib
import numpy as np
import pandas as pd
import psycopg2
import pytz
from tqdm import tqdm
from scipy.stats import norm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [2]:
weights_chan_df = joblib.load("weights_chan_df.pkl")
weights_acc_df = joblib.load("weights_acc_df.pkl")

In [5]:
weights_chan_df[weights_chan_df['account_id'] == 'bfcbb283-4e53-41fd-9b31-aa818f7f23ce']

,sales_channel_id,account_id,weights
241,1,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
242,3,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
243,4,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
244,5,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
245,6,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
246,7,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
247,8,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
248,9,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,0.0
249,ALL,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,1.0


In [ ]:
weights_acc_df['bfcbb283-4e53-41fd-9b31-aa818f7f23ce']

np.float64(0.0010090817356205853)

In [2]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [3]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [6]:
SAO_PAULO_TZ = pytz.timezone('America/Sao_Paulo')
LOOKBACK = 1
# SEASONAL_PERIODS = (24, 24*7)
# COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
# START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
START_DATE = END_DATE - timedelta(hours=23)
# TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [7]:
df.drop('count', axis=1, inplace=True)
for account_id in df['account_id'].unique():
    df = pd.concat([df, pd.DataFrame({'account_id': [account_id], 'sales_channel_id': ['ALL']})], ignore_index=True)

In [8]:
id_pairs = list(zip(df['account_id'], df['sales_channel_id']))

In [78]:
account_id = 'bfcbb283-4e53-41fd-9b31-aa818f7f23ce'
sales_channel_id = '1'

In [79]:
if sales_channel_id == 'ALL':
    cond = (sales['account_id'] == account_id) & \
           (sales['status'].notna())
else:
    sales_channel_id = int(sales_channel_id)
    cond = (sales['account_id'] == account_id) & \
           (sales['sales_channel_id'] == sales_channel_id) & \
           (sales['status'].notna())

In [80]:
df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
df_client = df_client.sort_values('created_date').reset_index(drop=True)
df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

In [81]:
df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

In [82]:
end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
start_date = end_date - relativedelta(months=LOOKBACK)
date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')

In [83]:
df_client_mod

,price_total_agg,n_orders
created_date,,
2025-03-04 08:00:00,48.49,1.0
2025-03-04 09:00:00,NaN,NaN
2025-03-04 10:00:00,66.64,1.0
2025-03-04 11:00:00,218.07,2.0
2025-03-04 12:00:00,NaN,NaN
...,...,...
2025-04-04 04:00:00,NaN,NaN
2025-04-04 05:00:00,NaN,NaN
2025-04-04 06:00:00,60.39,1.0


In [84]:
df = df_client_mod['n_orders'].fillna(0)
df = df.loc[df.index < START_DATE].copy()

In [87]:
df.median()

np.float64(0.0)

In [2]:
norm.ppf(0.95)

np.float64(1.644853626951472)

In [2]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [19]:
cursor.execute("select distinct account_id, channel from public.forecast where model='TBATS_10';")
res_tbats = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='GradientBoosting_10';")
res_gb = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='LSTM_10';")
res_lstm = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Ensemble_10';")
res_ens = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)';")
res_chronos1 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='Chronos_10';")
res_chronos2 = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model='MEDIAN_MAD_60';")
res_median = cursor.fetchall()

cursor.execute("select distinct account_id, channel from public.forecast where model like 'SARIMAX_%_10';")
res_sarimax = cursor.fetchall()

In [42]:
id_pairs = (
    set(res_tbats)
    .intersection(set(res_gb))
    .intersection(set(res_lstm))
    .intersection(set(res_ens))
    .intersection(set(res_chronos1))
    .intersection(set(res_chronos2))
    .intersection(set(res_median))
    .intersection(set(res_sarimax))
)

# id_pairs = set(res_tbats).intersection(set(res_gb))

In [12]:
len(id_pairs)

44

In [3]:
cursor.execute("select distinct model from public.forecast where model like 'SARIMAX%'");

In [4]:
res = cursor.fetchall()

In [13]:
pd.Series(res).apply(lambda x: x[0].split('_')).apply(lambda x: x[1]).value_counts().sort_index()

(0,0,0,0,0,0,0)     9
(0,0,0,0,0,0,24)    9
(0,0,0,0,0,1,24)    9
(0,0,0,0,0,2,24)    9
(0,0,0,1,0,1,24)    9
(0,0,0,1,0,2,24)    9
(0,0,0,2,0,1,24)    9
(0,0,1,2,0,0,24)    9
(0,1,0,0,0,1,24)    9
(0,1,0,1,0,0,24)    9
(0,1,1,0,0,2,24)    9
(1,0,0,0,0,0,24)    9
(1,0,0,0,0,1,24)    9
(1,0,0,1,0,1,24)    9
(1,0,0,2,0,1,24)    9
(1,0,1,0,0,1,24)    9
(1,0,1,2,0,0,24)    9
(1,0,2,0,0,2,24)    2
(1,0,2,2,0,1,24)    7
(2,0,0,0,0,1,24)    7
(2,0,0,0,0,2,24)    9
(2,0,0,2,0,0,24)    9
(2,0,0,2,0,1,24)    9
(2,0,1,0,0,0,24)    9
(2,0,2,0,0,0,24)    2
(2,0,2,1,0,0,24)    9
(2,0,2,1,0,1,24)    9
(2,1,0,2,0,0,24)    9
Name: count, dtype: int64